# DuckDB no Notisbook

Este exemplo usa o kernel DuckDB e o catálogo `notislake` para criar uma tabela Iceberg gerenciada, gravar um Parquet no lake e lê-lo de volta.

O schema precisa existir antes da tabela: o catálogo não o cria sozinho. `CREATE SCHEMA IF NOT EXISTS` o cria sem `LOCATION`, no bucket `notisbook-managed`, e permite reexecutar a célula.

In [ ]:
CREATE SCHEMA IF NOT EXISTS notislake.bronze

Criamos a tabela Iceberg `notislake.bronze.duckdb_employees` e a preenchemos com oito linhas escritas com `VALUES`. Sem `LOCATION`, a tabela é gerenciada e seus arquivos ficam em `notisbook-managed/bronze/duckdb_employees`. Para poder reexecutar, usamos `CREATE TABLE IF NOT EXISTS` e `DELETE` antes do `INSERT`, em vez de `DROP TABLE`: o `DROP` de uma tabela gerenciada só enfileira a pasta, quem a apaga é um kernel Spark ou Glue, e recriar a tabela antes disso é recusado (`pending_cleanup`).

In [ ]:
CREATE TABLE IF NOT EXISTS notislake.bronze.duckdb_employees (id BIGINT, nome VARCHAR, area VARCHAR);

DELETE FROM notislake.bronze.duckdb_employees;

INSERT INTO notislake.bronze.duckdb_employees
SELECT id::BIGINT AS id, nome, area
FROM (VALUES
    (1, 'Romerito', 'Platform Engineering'),
    (2, 'Theodoro', 'Data Engineering'),
    (3, 'Miguel', 'AI Engineering'),
    (4, 'Nina', 'Coffee Engineering'),
    (5, 'Niko', 'Security Engineering'),
    (6, 'Leonor', 'Governance'),
    (7, 'Ana', 'Data Engineering'),
    (8, 'Bruno', 'Platform Engineering')
) AS t(id, nome, area)

Exportamos a tabela para um arquivo Parquet na zona `landing` do notislake, acessada por path `s3a://`.

In [ ]:
COPY (SELECT * FROM notislake.bronze.duckdb_employees) TO 's3a://notislake/landing/duckdb_employees.parquet' (FORMAT parquet)

Lemos o Parquet de volta direto do path, sem passar pelo catálogo.

In [ ]:
SELECT * FROM read_parquet('s3a://notislake/landing/duckdb_employees.parquet')

Por fim, consultamos a tabela Iceberg pelo catálogo.

In [ ]:
SELECT * FROM notislake.bronze.duckdb_employees